# `derived_8.4-ece-model-salvage-1.1`

This report evaluates six retrained routing/model families at 60 selected features and the global model at 40, 50, 60, and 69 selected features after removing every SMAP-derived input. All selector, router, and model fitting is restricted to the seven Washington training stations; ECE targets are held out for spatial evaluation only.

## Reproducible setup

The next cell loads the tracked runner and generated artifacts so the notebook remains a reporting layer rather than a second implementation of the experiment.

In [1]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd

EXP_DIR = Path("experiment/derived_8.4-ece-model-salvage-1.1").resolve()
if not (EXP_DIR / "run_model_salvage.py").exists():
    EXP_DIR = Path.cwd().resolve()
spec = importlib.util.spec_from_file_location("model_salvage_11", EXP_DIR / "run_model_salvage.py")
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
config = runner.load_configuration()
predictions = pd.read_csv(EXP_DIR / "predictions.csv", low_memory=False)
seed_metrics = pd.read_csv(EXP_DIR / "seed_metrics.csv", low_memory=False)
summary = pd.read_csv(EXP_DIR / "summary.csv", low_memory=False)
audit = pd.read_csv(EXP_DIR / "routing_audit.csv", low_memory=False)
print(f"Experiment: {config['experiment']['name']}")
print(f"Models: {len(config['models'])}; feature sizes: {config['selected_feature_sizes']}; seeds: {config['seeds']}")
print(f"Prediction rows: {len(predictions):,}")

Experiment: derived_8.4-ece-model-salvage-1.1
Models: 10; feature sizes: [40, 50, 60, 69]; seeds: [42, 7, 13]
Prediction rows: 207,750


## Feature-selection provenance

The local selector is a copied and adapted MI → ElasticNet → stability → wrapper pipeline. Its accepted wrapper objective uses the WA 2023–2025 evaluation period as a documented manual-selection analogue. One evidence-ranked wrapper result is normalized into nested 40/50/60/69 feature manifests, with no specialist or delta additions.

In [2]:
print("REPORT_BEGIN::SELECTION")
selection_path = EXP_DIR / "feature_selection_artifacts" / "selected_features_by_size.json"
selection = json.loads(selection_path.read_text(encoding="utf-8"))
selected_by_size = selection["feature_sizes"]
candidate_pool = pd.read_csv(EXP_DIR / "feature_selection_artifacts" / "candidate_pool.csv")
selection_table = pd.DataFrame([
    {
        "status": selection.get("status"),
        "selected_features": int(size),
        "smap_features": sum("smap" in feature.lower() for feature in record["features"]),
        "candidate_pool": len(candidate_pool),
        "delta_additions": "none",
        "selection_period": "WA 2023–2025",
        "fit_scope": "WA only",
    }
    for size, record in sorted(selected_by_size.items(), key=lambda item: int(item[0]))
])
print(selection_table.to_markdown(index=False))
for size, record in sorted(selected_by_size.items(), key=lambda item: int(item[0])):
    print()
    print(f"### Selected feature manifest ({size} features)")
    print()
    print("```text")
    print("\n".join(record["features"]))
    print("```")
print("REPORT_END::SELECTION")

REPORT_BEGIN::SELECTION
| status   |   selected_features |   smap_features |   candidate_pool | delta_additions   | selection_period   | fit_scope   |
|:---------|--------------------:|----------------:|-----------------:|:------------------|:-------------------|:------------|
| complete |                  40 |               0 |               69 | none              | WA 2023–2025       | WA only     |
| complete |                  50 |               0 |               69 | none              | WA 2023–2025       | WA only     |
| complete |                  60 |               0 |               69 | none              | WA 2023–2025       | WA only     |
| complete |                  69 |               0 |               69 | none              | WA 2023–2025       | WA only     |

### Selected feature manifest (40 features)

```text
longitude
precip_mm
s2_b4
s2_b8
elev
slope
DOY
D_sin_DOY
D_cos_DOY
E_SAR_ratio
G_API
G_DSLR
G_rain_sum_3d
G_rain_sum_7d
V_rollrng_G_API_kobs14
V_rollmax_G_API_k

## Input and feature audit

This audit records the exact training/evaluation populations, the nested selected feature manifests, and the lineage-specific router inputs after SMAP filtering.

In [3]:
print("REPORT_BEGIN::INPUT_AUDIT")
with (EXP_DIR / "input_audit.json").open(encoding="utf-8") as handle:
    print(json.dumps(json.load(handle), indent=2, sort_keys=True))
print("REPORT_END::INPUT_AUDIT")

print("REPORT_BEGIN::FEATURE_AUDIT")
with (EXP_DIR / "feature_manifest.json").open(encoding="utf-8") as handle:
    feature_manifest = json.load(handle)
feature_rows = []
for name, value in feature_manifest.items():
    if name == "selected_model_feature_sets":
        for size, selected_value in sorted(value.items(), key=lambda item: int(item[0])):
            feature_rows.append({
                "component": f"selected_model_{size}",
                "parent_count": "selector",
                "dropped_smap": 0,
                "effective_count": len(selected_value["effective"]),
                "effective_features": ";".join(selected_value["effective"]),
            })
    elif isinstance(value, dict) and "effective" in value:
        feature_rows.append({
            "component": name,
            "parent_count": len(value["parent"]) if isinstance(value["parent"], list) else value["parent"],
            "dropped_smap": len(value["dropped"]) if isinstance(value["dropped"], list) else 0,
            "effective_count": len(value["effective"]),
            "effective_features": ";".join(value["effective"]),
        })
feature_table = pd.DataFrame(feature_rows)
print(feature_table.to_markdown(index=False))
print("REPORT_END::FEATURE_AUDIT")

REPORT_BEGIN::INPUT_AUDIT
{
  "ece_date_max": "2026-08-19",
  "ece_date_min": "2026-07-20",
  "ece_rows": 150,
  "ece_stations": [
    "ECE_BBG_Lost_Meadow",
    "ECE_BBG_Main_St",
    "ECE_Renton_Garden_North",
    "ECE_Renton_Garden_Shed",
    "ECE_Renton_Home"
  ],
  "ece_target_used_for_fit": false,
  "train_rows": 9803,
  "trainval_rows": 14608,
  "val_rows": 4805,
  "wa_test_rows": 6620,
  "wa_train_stations": [
    "BeaverPass_WA_990",
    "CayusePass_WA",
    "Darrington",
    "Paradise_WA",
    "Quinault",
    "SourdoughGulch_WA_985",
    "Spokane"
  ]
}
REPORT_END::INPUT_AUDIT
REPORT_BEGIN::FEATURE_AUDIT
| component              | parent_count   |   dropped_smap |   effective_count | effective_features                                                                                                                                                                                                                                                                                       

## Router audit

Regime shares are shown for WA trainval, the held-out WA temporal test, and the unseen ECE sensor set. Router seed remains fixed at 42.

In [4]:
print("REPORT_BEGIN::ROUTER_AUDIT")
print(audit.to_markdown(index=False, floatfmt=".4f"))
print("REPORT_END::ROUTER_AUDIT")

REPORT_BEGIN::ROUTER_AUDIT
| model_id                            | router       | dataset     |   router_seed |     n |   regime_0_share |   regime_1_share |   router_feature_count | router_features                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

## Ten-variant metrics

RMSE is the primary error metric; Pearson and first-difference Pearson correlation assess whether predictions follow the target level and temporal trend.

In [5]:
pooled = summary[summary["scope"] == "__pooled__"].copy()
columns = ["model_id", "dataset", "window", "n_seeds", "rmse_mean", "rmse_std", "mae_mean", "bias_mean", "ubrmse_mean", "r2_mean", "pearson_mean", "target_std_mean", "prediction_std_mean", "diff_pearson_mean"]
print("REPORT_BEGIN::METRICS")
print(pooled[columns].sort_values(["dataset", "window", "rmse_mean"]).to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::METRICS")

REPORT_BEGIN::METRICS
| model_id                            | dataset     | window               |   n_seeds |   rmse_mean |   rmse_std |   mae_mean |   bias_mean |   ubrmse_mean |   r2_mean |   pearson_mean |   target_std_mean |   prediction_std_mean |   diff_pearson_mean |
|:------------------------------------|:------------|:---------------------|----------:|------------:|-----------:|-----------:|------------:|--------------:|----------:|---------------:|------------------:|----------------------:|--------------------:|
| Global_Single_60_no_smap_fs60       | ece_spatial | spatial_ece_v3_full  |         3 |    0.050297 |   0.000506 |   0.042203 |    0.013950 |      0.048314 | -0.142906 |       0.008839 |          0.047049 |              0.011419 |           -0.014801 |
| Global_Single_69_no_smap_fs69       | ece_spatial | spatial_ece_v3_full  |         3 |    0.050380 |   0.000171 |   0.040536 |    0.008904 |      0.049554 | -0.146605 |      -0.137546 |          0.047049 |         

## Comparison with original SMAP-trained models

The original formal-evaluation results remain reference-only and are never used for 1.1 fitting. To keep this section compact, each model/split row reports the mean over the common three learner seeds; the seed-level paired artifact remains available for audit.

In [6]:
comparison = pd.read_csv(EXP_DIR / "reference_comparison.csv", low_memory=False)
print("REPORT_BEGIN::REFERENCE_COMPARISON")
if comparison.empty:
    print("No reference rows available.")
else:
    reference_mean = runner.summarize_reference_comparison(
        comparison, expected_seeds=config["seeds"]
    )
    columns = ["model_id", "dataset", "window", "n_seeds", "rmse_no_smap", "rmse_original", "rmse_delta_no_smap_minus_original", "pearson_no_smap", "pearson_original"]
    print(reference_mean[columns].to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::REFERENCE_COMPARISON")

REPORT_BEGIN::REFERENCE_COMPARISON
| model_id                            | dataset     | window              |   n_seeds |   rmse_no_smap |   rmse_original |   rmse_delta_no_smap_minus_original |   pearson_no_smap |   pearson_original |
|:------------------------------------|:------------|:--------------------|----------:|---------------:|----------------:|------------------------------------:|------------------:|-------------------:|
| Clustering_Backbone_k2_no_smap_fs60 | ece_spatial | spatial_ece_v3_full |         3 |       0.074408 |        0.058476 |                            0.015931 |         -0.299911 |           0.108498 |
| Clustering_Dynamic_k2_no_smap_fs60  | ece_spatial | spatial_ece_v3_full |         3 |       0.053308 |        0.058957 |                           -0.005649 |         -0.132648 |          -0.135071 |
| Clustering_V0_Full_k2_no_smap_fs60  | ece_spatial | spatial_ece_v3_full |         3 |       0.086613 |        0.058476 |                            0.02813

## Comparison of 1.1 selected features with 1.0 no-SMAP models

This same-seed comparison isolates the feature-selection change from the original SMAP-removal change. It is populated when the completed 1.0 prediction artifacts are available.

In [7]:
comparison_10 = pd.read_csv(EXP_DIR / "salvage_1_1_vs_1_0_summary.csv", low_memory=False)
print("REPORT_BEGIN::SALVAGE_1_1_VS_1_0")
if comparison_10.empty:
    print("No completed 1.0 paired prediction artifacts available yet.")
else:
    columns = ["model_id", "old_model_id", "dataset", "window", "n_seeds", "change_rmse_mean", "change_mae_mean", "change_bias_mean", "change_pearson_mean", "change_diff_pearson_mean"]
    print(comparison_10[columns].sort_values(["dataset", "window", "change_rmse_mean"]).to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::SALVAGE_1_1_VS_1_0")

REPORT_BEGIN::SALVAGE_1_1_VS_1_0
| model_id                            | old_model_id                     | dataset     | window              |   n_seeds |   change_rmse_mean |   change_mae_mean |   change_bias_mean |   change_pearson_mean |   change_diff_pearson_mean |
|:------------------------------------|:---------------------------------|:------------|:--------------------|----------:|-------------------:|------------------:|-------------------:|----------------------:|---------------------------:|
| Clustering_Backbone_k2_no_smap_fs60 | Clustering_Backbone54_k2_no_smap | ece_spatial | spatial_ece_v3_full |         3 |          -0.054169 |         -0.058383 |          -0.087876 |             -0.338503 |                   0.050146 |
| Clustering_V0_Full_k2_no_smap_fs60  | Clustering_V0_Full_k2_no_smap    | ece_spatial | spatial_ece_v3_full |         3 |          -0.046683 |         -0.048369 |          -0.055386 |              0.223366 |                   0.016765 |
| Global_Single

## Before vs after the new feature-selection round

This global-only comparison treats the 1.0 no-SMAP global model as the before-selection baseline and the four nested 1.1 global feature sizes as after-selection variants. It reports pooled ECE benefit, pooled WA degradation, and changes in level and first-difference trend correlation over the common seeds.

In [8]:
feature_selection_effect = pd.read_csv(EXP_DIR / "feature_selection_round_effect_summary.csv", low_memory=False)
print("REPORT_BEGIN::FEATURE_SELECTION_ROUND")
if feature_selection_effect.empty:
    print("Feature-selection comparison is not available yet.")
else:
    table_rows = []
    for feature_size in sorted(feature_selection_effect["feature_size"].unique()):
        ece_row = feature_selection_effect[
            (feature_selection_effect["feature_size"] == feature_size)
            & (feature_selection_effect["split"] == "ECE spatial")
        ]
        wa_row = feature_selection_effect[
            (feature_selection_effect["feature_size"] == feature_size)
            & (feature_selection_effect["split"] == "WA temporal")
        ]
        if len(ece_row) != 1 or len(wa_row) != 1:
            raise ValueError(f"Expected one ECE and one WA row for {feature_size} features.")
        ece_row = ece_row.iloc[0]
        wa_row = wa_row.iloc[0]
        table_rows.append({
            "features": int(feature_size),
            "n_seeds": int(ece_row["n_seeds"]),
            "ECE before RMSE": ece_row["rmse_before_mean"],
            "ECE after RMSE": ece_row["rmse_after_mean"],
            "ECE benefit": ece_row["rmse_effect_mean"],
            "ECE benefit %": ece_row["rmse_effect_pct_mean"],
            "ECE ΔPearson": ece_row["pearson_change_mean"],
            "ECE Δdiff Pearson": ece_row["diff_pearson_change_mean"],
            "WA before RMSE": wa_row["rmse_before_mean"],
            "WA after RMSE": wa_row["rmse_after_mean"],
            "WA degradation": wa_row["rmse_effect_mean"],
            "WA degradation %": wa_row["rmse_effect_pct_mean"],
            "WA ΔPearson": wa_row["pearson_change_mean"],
            "WA Δdiff Pearson": wa_row["diff_pearson_change_mean"],
        })
    print(pd.DataFrame(table_rows).to_markdown(index=False, floatfmt=".6f"))
    print("INTERPRETATION_BEGIN")
    print(runner.interpret_feature_selection_round(feature_selection_effect, expected_seeds=(42, 7, 13)))
    print("INTERPRETATION_END")
print("REPORT_END::FEATURE_SELECTION_ROUND")

REPORT_BEGIN::FEATURE_SELECTION_ROUND
|   features |   n_seeds |   ECE before RMSE |   ECE after RMSE |   ECE benefit |   ECE benefit % |   ECE ΔPearson |   ECE Δdiff Pearson |   WA before RMSE |   WA after RMSE |   WA degradation |   WA degradation % |   WA ΔPearson |   WA Δdiff Pearson |
|-----------:|----------:|------------------:|-----------------:|--------------:|----------------:|---------------:|--------------------:|-----------------:|----------------:|-----------------:|-------------------:|--------------:|-------------------:|
|  40.000000 |  3.000000 |          0.057245 |         0.051302 |      0.005943 |       10.370269 |       0.077461 |           -0.024842 |         0.050591 |        0.048867 |        -0.001724 |          -3.407353 |      0.012996 |          -0.023409 |
|  50.000000 |  3.000000 |          0.057245 |         0.050713 |      0.006532 |       11.398415 |       0.103339 |            0.010294 |         0.050591 |        0.048722 |        -0.001869 |         

## Effect of Removing SMAP: ECE Benefit vs WA Degradation

This paired summary compares 1.1 against the original SMAP-trained references. ECE benefit is original RMSE minus no-SMAP RMSE; WA degradation is no-SMAP RMSE minus original RMSE. Global feature-count variants are listed separately. The effect chart uses a shared fixed y-axis of −0.04 to 0.04 RMSE.

In [9]:
effect_summary = pd.read_csv(EXP_DIR / "old_vs_new_effect_summary.csv", low_memory=False)
effect_columns = ["model_id", "split", "n_seeds", "rmse_original_mean", "rmse_no_smap_mean", "effect_rmse_mean", "effect_rmse_std", "effect_rmse_pct_mean", "improved_seeds", "worsened_seeds", "pearson_change_mean", "diff_pearson_change_mean"]
print("REPORT_BEGIN::OLD_NEW_EFFECT")
if effect_summary.empty:
    print("No paired old-vs-new rows available.")
else:
    print(effect_summary[effect_columns].sort_values(["split", "effect_rmse_mean"], ascending=[True, False]).to_markdown(index=False, floatfmt=".6f"))
effect_figure = runner.make_old_vs_new_effect_figure(effect_summary, EXP_DIR / "figures")
print(f"FIGURE::{effect_figure.name}")
print("REPORT_END::OLD_NEW_EFFECT")

REPORT_BEGIN::OLD_NEW_EFFECT
| model_id                            | split       |   n_seeds |   rmse_original_mean |   rmse_no_smap_mean |   effect_rmse_mean |   effect_rmse_std |   effect_rmse_pct_mean |   improved_seeds |   worsened_seeds |   pearson_change_mean |   diff_pearson_change_mean |
|:------------------------------------|:------------|----------:|---------------------:|--------------------:|-------------------:|------------------:|-----------------------:|-----------------:|-----------------:|----------------------:|---------------------------:|
| Global_Single_60_no_smap_fs60       | ECE spatial |         3 |             0.059287 |            0.050297 |           0.008991 |          0.000449 |              15.162487 |                3 |                0 |              0.067340 |                        nan |
| Global_Single_69_no_smap_fs69       | ECE spatial |         3 |             0.059287 |            0.050380 |           0.008908 |          0.000763 |              15

FIGURE::old_vs_new_rmse_effect.png
REPORT_END::OLD_NEW_EFFECT


## SMAP-invariance check

Seed-42 ECE rows are evaluated once with native SMAP values and once with all SMAP columns replaced by zero; a no-SMAP model must produce identical regimes and predictions.

In [10]:
invariance = pd.read_csv(EXP_DIR / "smap_invariance.csv", low_memory=False)
print("REPORT_BEGIN::SMAP_INVARIANCE")
print(invariance.to_markdown(index=False, floatfmt=".12f"))
print("REPORT_END::SMAP_INVARIANCE")

REPORT_BEGIN::SMAP_INVARIANCE
| model_id                            |   seed |   smap_columns_altered |   max_abs_prediction_difference |   changed_regime_labels |
|:------------------------------------|-------:|-----------------------:|--------------------------------:|------------------------:|
| Clustering_V0_Full_k2_no_smap_fs60  |     42 |                     85 |                  0.000000000000 |                       0 |
| Clustering_Backbone_k2_no_smap_fs60 |     42 |                     85 |                  0.000000000000 |                       0 |
| Trained_Gating_k2_no_smap_fs60      |     42 |                     85 |                  0.000000000000 |                       0 |
| Univariate_G_API_k2_no_smap_fs60    |     42 |                     85 |                  0.000000000000 |                       0 |
| Clustering_Dynamic_k2_no_smap_fs60  |     42 |                     85 |                  0.000000000000 |                       0 |
| Seasonal_Binary_k2_no_smap_fs6

## Global model version comparison

Each ECE station receives one seven-line chart comparing the original global model, the 1.0 no-SMAP global model, the 1.1 global models at 40/50/60/69 features, and ground truth. Predictions are aligned by station/date and averaged over the common seeds `[42, 7, 13]`. Every ECE line chart uses the same fixed y-axis of 0.00 to 0.25 soil-moisture units.

In [11]:
print("REPORT_BEGIN::GLOBAL_VERSION")
version_data = runner.load_data(config)
version_status = runner.global_version_source_status(version_data, config, (42, 7, 13))
print(json.dumps(version_status, indent=2, sort_keys=True))
if version_status["ready"]:
    version_paths = runner.make_global_version_charts(version_data, config, EXP_DIR / "figures", (42, 7, 13))
    for path in version_paths:
        print(f"FIGURE::{path.name}")
else:
    print("Global-version figures deferred until all old, 1.0, and 1.1 common-seed artifacts are complete.")
print("REPORT_END::GLOBAL_VERSION")

REPORT_BEGIN::GLOBAL_VERSION


{
  "ece_dates_per_station": 30,
  "ece_rows": 150,
  "ece_stations": [
    "ECE_BBG_Lost_Meadow",
    "ECE_BBG_Main_St",
    "ECE_Renton_Garden_North",
    "ECE_Renton_Garden_Shed",
    "ECE_Renton_Home"
  ],
  "missing": [],
  "original_paths": [
    "/scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-formal-eval-2.1-ece-v3/predictions_spatial/Global_Single_54__s42__ece_preds.npy",
    "/scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-formal-eval-2.1-ece-v3/predictions_spatial/Global_Single_54__s7__ece_preds.npy",
    "/scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-formal-eval-2.1-ece-v3/predictions_spatial/Global_Single_54__s13__ece_preds.npy"
  ],
  "ready": true,
  "salvage_1_0_paths": [
    "/scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-ece-model-salvage-1.0/artifacts/predictions/Global_Single_54_no_smap__s42.csv",
    "/scratch/group/p.cis250607.000/MDR-Project/notebook

FIGURE::ece_ECE_BBG_Lost_Meadow_global_model_versions.png
FIGURE::ece_ECE_BBG_Main_St_global_model_versions.png
FIGURE::ece_ECE_Renton_Garden_North_global_model_versions.png
FIGURE::ece_ECE_Renton_Garden_Shed_global_model_versions.png
FIGURE::ece_ECE_Renton_Home_global_model_versions.png
REPORT_END::GLOBAL_VERSION


## Trend figures

The next cell generates three figure suites per ECE station: architecture gates, alternative regime gates, and all four global feature sizes. Each chart is limited to five lines including the observed target and uses the common fixed y-axis of 0.00 to 0.25 soil-moisture units.

In [12]:
figure_paths = runner.make_trend_figures(predictions, EXP_DIR / "figures")
print("REPORT_BEGIN::FIGURES")
for path in figure_paths:
    print(f"- {path.name}")
print("REPORT_END::FIGURES")

REPORT_BEGIN::FIGURES
- ece_ECE_BBG_Lost_Meadow_architecture_trend.png
- ece_ECE_BBG_Lost_Meadow_regime_trend.png
- ece_ECE_BBG_Lost_Meadow_global_feature_sizes_trend.png
- ece_ECE_BBG_Main_St_architecture_trend.png
- ece_ECE_BBG_Main_St_regime_trend.png
- ece_ECE_BBG_Main_St_global_feature_sizes_trend.png
- ece_ECE_Renton_Garden_North_architecture_trend.png
- ece_ECE_Renton_Garden_North_regime_trend.png
- ece_ECE_Renton_Garden_North_global_feature_sizes_trend.png
- ece_ECE_Renton_Garden_Shed_architecture_trend.png
- ece_ECE_Renton_Garden_Shed_regime_trend.png
- ece_ECE_Renton_Garden_Shed_global_feature_sizes_trend.png
- ece_ECE_Renton_Home_architecture_trend.png
- ece_ECE_Renton_Home_regime_trend.png
- ece_ECE_Renton_Home_global_feature_sizes_trend.png
REPORT_END::FIGURES


## Multi-panel ECE sensor comparisons

Each model group is also shown as one multi-panel image containing all five ECE sensors. Panels share the fixed 0.00–0.25 soil-moisture y-axis, while the individual trend figures above remain available for detailed inspection.

In [13]:
multipanel_paths = runner.make_multipanel_trend_figures(predictions, EXP_DIR / "figures")
print("REPORT_BEGIN::MULTIPANEL")
for path in multipanel_paths:
    print(f"FIGURE::{path.name}")
print("REPORT_END::MULTIPANEL")

REPORT_BEGIN::MULTIPANEL
FIGURE::ece_all_sensors_architecture_multipanel.png
FIGURE::ece_all_sensors_regime_multipanel.png
FIGURE::ece_all_sensors_global_feature_sizes_multipanel.png
REPORT_END::MULTIPANEL


## Original versus best 1.1 global validation

This focused diagnostic compares ground truth with the original global model and the 1.1 global model having the lowest pooled ECE RMSE across the common seeds. The unused bottom-right panel reports pooled ECE RMSE, MAE, Pearson correlation, and RMSE improvement. Similar trend shapes would support the hypothesis that SMAP availability drives the original-model consistency.

In [14]:
validation_data = runner.load_data(config)
best_global_path = runner.make_best_global_validation_multipanel(
    validation_data, predictions, summary, config, EXP_DIR / "figures", (42, 7, 13)
)
best_global_provenance = json.loads(
    (EXP_DIR / "global_best_validation_provenance.json").read_text(encoding="utf-8")
)
print("REPORT_BEGIN::BEST_GLOBAL_VALIDATION")
print(json.dumps(best_global_provenance, indent=2, sort_keys=True))
print(f"FIGURE::{best_global_path.name}")
print("REPORT_END::BEST_GLOBAL_VALIDATION")

REPORT_BEGIN::BEST_GLOBAL_VALIDATION
{
  "aggregation": "mean over chart seeds",
  "best_model_id": "Global_Single_60_no_smap_fs60",
  "best_pooled_ece_rmse": 0.0502967871673428,
  "dates_per_station": {
    "ECE_BBG_Lost_Meadow": 30,
    "ECE_BBG_Main_St": 30,
    "ECE_Renton_Garden_North": 30,
    "ECE_Renton_Garden_Shed": 30,
    "ECE_Renton_Home": 30
  },
  "ece_stations": [
    "ECE_BBG_Lost_Meadow",
    "ECE_BBG_Main_St",
    "ECE_Renton_Garden_North",
    "ECE_Renton_Garden_Shed",
    "ECE_Renton_Home"
  ],
  "line_count": 3,
  "line_labels": [
    "Ground truth",
    "Original Global_Single_54",
    "Best 1.1 (Global_Single_60_no_smap_fs60)"
  ],
  "pooled_ece_metrics": {
    "best_1_1": {
      "mae": 0.0422026438575099,
      "pearson": 0.0088391209018219,
      "rmse": 0.0502967871673428
    },
    "original": {
      "mae": 0.05118888073910786,
      "pearson": -0.05850091631388754,
      "rmse": 0.059287378229081276
    },
    "rmse_improvement": 0.008990591061738475,
    

## Completion

All tables above are sourced from executed cells, and the linked figures are generated during this notebook execution.

In [15]:
print("Notebook complete — all report sections above are generated from experiment artifacts.")

Notebook complete — all report sections above are generated from experiment artifacts.
